In [ ]:
# ============================================================
# Improved MIM + ViT Segmentation Pipeline (Binary, 2 classes)
# - Stronger decoder (UNet-like upsampling)
# - Focal + Dice loss
# - BCE + Dice loss option
# - Data augmentation (flip + small rotation)
# - Different LR for backbone vs decoder
# - Patch size 8 for more spatial detail
# - Test-Time Augmentation (TTA) at inference
# ============================================================

import os, random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

import matplotlib.pyplot as plt

import timm

# ----------------- Reproducibility --------------------------
SEED = 120225

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ----------------- Paths ------------------------------------
DATA_ROOT   = Path("/home/tali1/rat_brain_seg/data/new_dataset")
TRAIN_IMG   = DATA_ROOT / "train_images"
TRAIN_MASK  = DATA_ROOT / "train_masks"
VAL_IMG     = DATA_ROOT / "val_images"
VAL_MASK    = DATA_ROOT / "val_masks"
TEST_IMG    = DATA_ROOT / "test_images"
TEST_MASK   = DATA_ROOT / "test_masks"

MIM_OUT     = Path("mim_pretrained_2d_improved.09pth")
SEG_OUT     = Path("vit_segmentation_improved.pth")

# ----------------- Hyperparameters --------------------------
IMG_SIZE        = 256
PATCH_SIZE      = 8      # smaller patch for better detail
NUM_CLASSES     = 1
MASK_RATIO      = 0.5
EMBED_DIM       = 768

BATCH           = 4 

EPOCHS_MIM      = 50
EPOCHS_SEG      = 100

MIM_LR          = 1e-4
LR_BACKBONE     = 3e-5  # 1e-5 93%
LR_DECODER      = 3e-4
WEIGHT_DECAY    = 1e-4   #1e-4

PATIENCE        = 15        # early stopping patience



# ============================================================
#                     DATASETS
# ============================================================

class TIFF2DDataset(Dataset):
    """Single-image dataset for MIM pretraining."""
    def __init__(self, root, img_size=IMG_SIZE):
        self.paths = sorted([
            p for p in Path(root).iterdir()
            if p.suffix.lower() in [".tif", ".tiff", ".png", ".jpg", ".jpeg"]
        ])
        self.resize = transforms.Resize((img_size, img_size))
        self.to_tensor = transforms.ToTensor()
        self.img_size = img_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        im = Image.open(self.paths[i]).convert("L")
        im = self.resize(im)
        x = self.to_tensor(im)  # [1,H,W]

        # per-image normalization
        x = (x - x.mean()) / (x.std() + 1e-6)
        return x


class PairDataset(Dataset):
    """Image-mask pair dataset for segmentation."""
    def __init__(self, images, masks, img_size=IMG_SIZE, augment=False):
        self.images = sorted([
            p for p in Path(images).iterdir()
            if p.suffix.lower() in [".tif", ".tiff", ".png", ".jpg", ".jpeg"]
        ])
        self.masks_dir = Path(masks)
        self.img_size = img_size
        self.augment = augment

        self.resize_img = transforms.Resize((img_size, img_size), interpolation=InterpolationMode.BILINEAR)
        self.resize_mask = transforms.Resize((img_size, img_size), interpolation=InterpolationMode.NEAREST)

    def __len__(self):
        return len(self.images)

    def _load_mask(self, ip: Path):
        stem = ip.stem
        for ext in [".png", ".tif", ".tiff", ".jpg", ".jpeg"]:
            mf = self.masks_dir / (stem + ext)
            if mf.exists():
                return Image.open(mf).convert("L")
        raise FileNotFoundError(f"Mask not found for image {ip}")

    def _random_flip_rotate(self, img, mask):
        """Apply same random flip/rotation to image and mask."""
        if random.random() < 0.5:
            img = TF.hflip(img)
            mask = TF.hflip(mask)
        if random.random() < 0.5:
            img = TF.vflip(img)
            mask = TF.vflip(mask)
        # small random rotation
        if random.random() < 0.5:
            angle = random.uniform(-10, 10)
            img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
        return img, mask

    def __getitem__(self, i):
        ip = self.images[i]
        img = Image.open(ip).convert("L")
        mask = self._load_mask(ip)

        # resize
        img = self.resize_img(img)
        mask = self.resize_mask(mask)

        # augment if training
        if self.augment:
            img, mask = self._random_flip_rotate(img, mask)

        # to tensor
        img = TF.to_tensor(img)  # [1,H,W]
        # per-image normalization
        img = (img - img.mean()) / (img.std() + 1e-6)

        mask_np = np.array(mask)
        mask_bin = (mask_np > 127).astype(np.int64)  # 0/1
        mask_t = torch.from_numpy(mask_bin)          # [H,W] long

        return img, mask_t


def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

# ============================================================
#                     MODELS
# ============================================================

class ViTBackbone(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch=PATCH_SIZE, model_name="vit_base_patch16_224"):
        super().__init__()
        self.embed_dim = EMBED_DIM
        self.patch = patch
        self.grid = img_size // patch

        # custom 1-channel patch embedding
        self.proj = nn.Conv2d(1, self.embed_dim, kernel_size=patch, stride=patch)

        # learnable positional embedding
        self.pos = nn.Parameter(torch.zeros(1, self.grid * self.grid, self.embed_dim))

        # use timm ViT blocks (pretrained weights on ImageNet)
        self.enc = timm.create_model(model_name, pretrained=True, num_classes=0)

        # we don't use its patch_embed; freeze it to be safe
        if hasattr(self.enc, "patch_embed"):
            for p in self.enc.patch_embed.parameters():
                p.requires_grad = False

    def forward(self, x):
        # x: [B,1,H,W]
        t = self.proj(x)                           # [B,C,H/ps,W/ps]
        B, C, H, W = t.shape
        t = t.flatten(2).transpose(1, 2)           # [B,N,C]
        t = t + self.pos

        # pass through transformer blocks & norm
        for blk in self.enc.blocks:
            t = blk(t)
        t = self.enc.norm(t)                       # [B,N,C]
        return t


class MIMHead(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, patch=PATCH_SIZE):
        super().__init__()
        self.patch = patch
        self.proj = nn.Linear(embed_dim, patch * patch)

    def forward(self, tokens):
        # tokens: [B,N,C] -> [B,N,P^2]
        return self.proj(tokens)


class SegDecoder(nn.Module):
    """Stronger UNet-like upsampling decoder."""
    def __init__(self, embed_dim=EMBED_DIM, img_size=IMG_SIZE, patch=PATCH_SIZE, num_classes=NUM_CLASSES):
        super().__init__()
        h = img_size // patch

        self.conv1 = nn.Sequential(
            nn.Conv2d(embed_dim, 512, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.GELU()
        )

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.block1 = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1),
            nn.GELU()
        )

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.GELU()
        )

        self.up3 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.GELU()
        )

        self.out = nn.Conv2d(32, num_classes, 1)

        self.h = h
        self.patch = patch
        self.img_size = img_size

    def forward(self, tokens):
        B, N, C = tokens.shape
        h = self.img_size // self.patch

        x = tokens.transpose(1, 2).reshape(B, C, h, h)  # [B,C,h,h]

        x = self.conv1(x)
        x = self.block1(self.up1(x))
        x = self.block2(self.up2(x))
        x = self.block3(self.up3(x))

        x = F.interpolate(x, size=(self.img_size, self.img_size),
                          mode="bilinear", align_corners=False)

        return self.out(x)


class SegDecoderrrrr(nn.Module):
    """Stronger UNet-like upsampling decoder with Dropout."""
    def __init__(self, embed_dim=EMBED_DIM, img_size=IMG_SIZE, patch=PATCH_SIZE, num_classes=NUM_CLASSES):
        super().__init__()
        h = img_size // patch

        # --- Block 1 ---
        self.conv1 = nn.Sequential(
            nn.Conv2d(embed_dim, 512, 3, padding=1),
            nn.GELU(),
            nn.Dropout2d(0.3),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.GELU(),
            nn.Dropout2d(0.3),
        )

        # --- Block 2 ---
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.block1 = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1),
            nn.GELU(),
            nn.Dropout2d(0.2),
        )

        # --- Block 3 ---
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.GELU(),
            nn.Dropout2d(0.1),
        )

        # --- Block 4 ---
        self.up3 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.GELU(),
            nn.Dropout2d(0.05),
        )

        # Output layer
        self.out = nn.Conv2d(32, num_classes, 1)

        self.h = h
        self.patch = patch
        self.img_size = img_size

    def forward(self, tokens):
        B, N, C = tokens.shape
        h = self.img_size // self.patch

        x = tokens.transpose(1, 2).reshape(B, C, h, h)

        x = self.conv1(x)
        x = self.block1(self.up1(x))
        x = self.block2(self.up2(x))
        x = self.block3(self.up3(x))

        x = F.interpolate(x, size=(self.img_size, self.img_size),
                          mode="bilinear", align_corners=False)

        return self.out(x)


# ============================================================
#                     LOSSES & METRICS
# ============================================================

class FocalDiceLoss(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.num_classes = num_classes

    def forward(self, logits, targets):
        """
        logits: [B,C,H,W]
        targets: [B,H,W] (long 0..C-1)
        """
        probs = torch.softmax(logits, dim=1)
        targets_1h = F.one_hot(targets, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        # Dice loss (class-wise)
        inter = (probs * targets_1h).sum((0, 2, 3))
        den   = probs.sum((0, 2, 3)) + targets_1h.sum((0, 2, 3)) + 1e-6
        dice_loss = 1.0 - (2.0 * inter / den).mean()

        # Focal Cross Entropy
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        focal_loss = ((1 - pt) ** self.gamma * ce).mean()

        return 0.5 * dice_loss + 0.5 * focal_loss


def dice_coef(logits, targets, eps=1e-6):
    """
    logits: [B, 1, H, W]  (raw model outputs)
    targets: [B, H, W]    (0/1 mask)
    returns: scalar Dice score in [0,1]
    """
    # Convert logits -> probabilities
    probs = torch.sigmoid(logits)          # [B,1,H,W]

    # Binarize at 0.5
    preds = (probs > 0.5).float()          # [B,1,H,W]

    # Make targets shape [B,1,H,W]
    targets = targets.float().unsqueeze(1) # [B,1,H,W]

    # Intersection and union
    inter = (preds * targets).sum(dim=(0,1,2,3))
    den   = preds.sum(dim=(0,1,2,3)) + targets.sum(dim=(0,1,2,3)) + eps

    dice = (2.0 * inter) / den            # in [0,1]
    return dice


class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # logits: [B,1,H,W]
        # targets: [B,H,W] (0/1)

        # use the single channel
        logits_fg = logits.squeeze(1)          # [B,H,W]

        # BCE
        bce_loss = self.bce(logits_fg, targets.float())

        # Dice
        probs = torch.sigmoid(logits_fg)       # [B,H,W]
        inter = (probs * targets).sum()
        den = probs.sum() + targets.sum() + 1e-6
        dice = 1 - (2.0 * inter / den)

        return 0.5 * bce_loss + 0.5 * dice


class DiceCELoss(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, target):
        # Cross-entropy loss
        ce = self.ce(logits, target)

        # Softmax to probs
        probs = torch.softmax(logits, dim=1)

        # One-hot encode target
        onehot = F.one_hot(target, num_classes=logits.shape[1]).permute(0,3,1,2).float()

        # Dice loss
        inter = (probs * onehot).sum((0,2,3))
        den = probs.sum((0,2,3)) + onehot.sum((0,2,3)) + 1e-6
        dice = 1 - (2 * inter + 1e-6) / den

        return 0.5 * ce + 0.5 * dice.mean()

# ============================================================
#                     MIM PRETRAINING
# ============================================================

def make_random_mask(B, N, mask_ratio):
    len_keep = int(N * (1 - mask_ratio))
    noise = torch.rand(B, N, device=DEVICE)
    ids_shuffle = torch.argsort(noise, dim=1)
    mask = torch.ones(B, N, device=DEVICE)
    mask.scatter_(1, ids_shuffle[:, :len_keep], 0)
    return mask.bool()


def save_masked_patches(x, mask, patch_size, out_dir):
    """
    Saves only the masked patches as separate images.
    """
    import os
    os.makedirs(out_dir, exist_ok=True)

    C, H, W = x.shape
    idx = 0
    saved = 0

    for i in range(0, H, patch_size):
        for j in range(0, W, patch_size):
            if mask[idx] == 1:
                patch = x[:, i:i+patch_size, j:j+patch_size]
                patch_img = transforms.ToPILImage()(patch.cpu())
                patch_img.save(f"{out_dir}/patch_{idx}.png")
                saved += 1
            idx += 1

    print(f"Saved {saved} masked patches → {out_dir}")


def run_mim_pretraining():
    print("\n========== MIM PRETRAINING ==========")
    dataset = TIFF2DDataset(TRAIN_IMG, IMG_SIZE)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH,
        shuffle=True,
        num_workers=2,
        worker_init_fn=worker_init_fn,
        pin_memory=True
    )

    vit  = ViTBackbone(img_size=IMG_SIZE, patch=PATCH_SIZE).to(DEVICE)
    head = MIMHead(embed_dim=vit.embed_dim, patch=PATCH_SIZE).to(DEVICE)

    opt = torch.optim.AdamW(
        list(vit.parameters()) + list(head.parameters()),
        lr=MIM_LR,
        weight_decay=WEIGHT_DECAY
    )

    mse = nn.MSELoss()
    N = (IMG_SIZE // PATCH_SIZE) * (IMG_SIZE // PATCH_SIZE)

    train_losses = []

    for ep in range(EPOCHS_MIM):
        vit.train()
        head.train()
        loss_sum = 0.0
        count = 0

        for x in dataloader:
            x = x.to(DEVICE)  # [B,1,H,W]

            patches = F.unfold(
                x,
                kernel_size=PATCH_SIZE,
                stride=PATCH_SIZE
            ).transpose(1, 2)         # [B,N,P^2]

            tokens = vit(x)           # [B,N,C]
            mask = make_random_mask(x.size(0), N, MASK_RATIO)  # [B,N]

            pred = head(tokens)[mask]   # [M,P^2]
            target = patches[mask]      # [M,P^2]
            loss = mse(pred, target)

            opt.zero_grad()
            loss.backward()
            opt.step()

            loss_sum += loss.item() * x.size(0)
            count += x.size(0)

        epoch_loss = loss_sum / max(count, 1)
        train_losses.append(epoch_loss)
        print(f"[MIM] epoch {ep+1}/{EPOCHS_MIM} | train_loss={epoch_loss:.6f}")

    # Plot MIM loss
    plt.figure(figsize=(7, 5))
    plt.plot(range(1, len(train_losses) + 1), train_losses, linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("MIM Pretraining: Training Loss Curve")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("mim_training_curve.png")
    plt.close()

    # Save checkpoint
    torch.save(
        {"vit": vit.state_dict(), "img_size": IMG_SIZE, "patch": PATCH_SIZE},
        MIM_OUT
    )
    print(f"✅ Saved pretrained model → {MIM_OUT}")
# ============================================================
#                     SEGMENTATION TRAINING
# ============================================================

def build_segmentation_models():
    vit = ViTBackbone(IMG_SIZE, PATCH_SIZE).to(DEVICE)
    if MIM_OUT.exists():
        ckpt = torch.load(MIM_OUT, map_location=DEVICE)
        vit.load_state_dict(ckpt["vit"], strict=False)
        print("Loaded MIM-pretrained ViT weights.")
    else:
        print("⚠️ MIM checkpoint not found. Training segmentation from scratch backbone.")

    dec = SegDecoder(vit.embed_dim, IMG_SIZE, PATCH_SIZE, NUM_CLASSES).to(DEVICE)
    return vit, dec


def run_segmentation_training():
    print("\n========== SEGMENTATION TRAINING ==========")

    train_ds = PairDataset(TRAIN_IMG, TRAIN_MASK, IMG_SIZE, augment=False)
    val_ds   = PairDataset(VAL_IMG, VAL_MASK, IMG_SIZE, augment=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=2,
        worker_init_fn=worker_init_fn,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=2,
        worker_init_fn=worker_init_fn,
        pin_memory=True
    )

    vit, dec = build_segmentation_models()

    
    # crit = FocalDiceLoss(NUM_CLASSES)  # for multi-class segmentation
    crit = BCEDiceLoss()   # for binary segmentation with BCE + Dice
    # crit = DiceCELoss(NUM_CLASSES) # alternative for multi-class segmentation

    # opt = torch.optim.AdamW(
    #     list(vit.parameters()) + list(dec.parameters()),
    #     lr=1e-3, weight_decay=5e-5
    # )
    opt = torch.optim.AdamW(
    [
        {"params": vit.parameters(), "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY},
        {"params": dec.parameters(), "lr": LR_DECODER, "weight_decay": WEIGHT_DECAY},
    ]
)
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    #     opt, T_0=10, T_mult=2, eta_min=1e-7
    # )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
    )




    best_val_loss = float("inf")
    epochs_no_improve = 0
    train_loss_curve, val_loss_curve = [], []

    for ep in range(EPOCHS_SEG):
        vit.train()
        dec.train()
        loss_sum = 0.0
        cnt = 0

        for x, y in train_loader:
            x = x.to(DEVICE)          # [B,1,H,W]
            y = y.to(DEVICE)          # [B,H,W]

            logits = dec(vit(x))      # [B,C,H,W]
            loss = crit(logits, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            loss_sum += loss.item() * x.size(0)
            cnt += x.size(0)

        avg_train_loss = loss_sum / max(cnt, 1)
        train_loss_curve.append(avg_train_loss)

        # ---- Validation ----
        vit.eval()
        dec.eval()
        val_loss_sum = 0.0
        val_cnt = 0
        dices = []

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE)
                y = y.to(DEVICE)

                logits = dec(vit(x))
                loss_val = crit(logits, y)
                
                val_loss_sum += loss_val.item() * x.size(0)
                val_cnt += x.size(0)

                dices.append(dice_coef(logits, y).item())

        avg_val_loss = val_loss_sum / max(val_cnt, 1)
        val_loss_curve.append(avg_val_loss)
        md = float(np.mean(dices)) if dices else 0.0
        scheduler.step(avg_val_loss)
        print(f"[SEG] ep {ep+1}/{EPOCHS_SEG} | train_loss={avg_train_loss:.4f} | "
              f"val_loss={avg_val_loss:.4f} | valDice={md:.4f}")

        # ---- Early stopping and checkpoint ----
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save({"vit": vit.state_dict(), "dec": dec.state_dict()}, SEG_OUT)
            print(f"✅ Saved best model (val_loss={best_val_loss:.4f}): {SEG_OUT}")
        else:
            epochs_no_improve += 1
            print(f"⚠️ No improvement for {epochs_no_improve} epochs.")

        if epochs_no_improve >= PATIENCE:
            print(f"⛔ Early stopping triggered after {ep+1} epochs! "
                  f"Best val_loss={best_val_loss:.4f}")
            break

    # Plot curves
    epochs_r = range(1, len(train_loss_curve) + 1)
    plt.figure(figsize=(9, 6))
    plt.plot(epochs_r, train_loss_curve, label="Train Loss")
    plt.plot(epochs_r, val_loss_curve, label="Val Loss")
    plt.legend()
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Segmentation Progress")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("segmentation_training_curve.png")
    plt.close()
    print("Saved training curves (MIM + segmentation).")


# ============================================================
#                     TEST / INFERENCE WITH TTA
# ============================================================

def predict_tta(vit, dec, x):
    """
    Simple Test-Time Augmentation:
    - original
    - horizontal flip
    """
    logits_list = []

    # original
    logits_o = dec(vit(x))
    logits_list.append(logits_o)

    # h-flip
    x_flip = torch.flip(x, dims=[3])
    logits_f = dec(vit(x_flip))
    logits_f = torch.flip(logits_f, dims=[3])
    logits_list.append(logits_f)

    logits_stack = torch.stack(logits_list, dim=0)  # [T,B,C,H,W]
    logits_mean = logits_stack.mean(dim=0)
    return logits_mean


def run_test():
    print("\n========== TESTING WITH TTA ==========")
    test_ds = PairDataset(TEST_IMG, TEST_MASK, IMG_SIZE, augment=False)
    test_dl = DataLoader(
        test_ds,
        batch_size=BATCH,
        shuffle=False,
        num_workers=2,
        worker_init_fn=worker_init_fn,
        pin_memory=True
    )

    ckpt = torch.load(SEG_OUT, map_location=DEVICE)
    vit = ViTBackbone(IMG_SIZE, PATCH_SIZE).to(DEVICE)
    dec = SegDecoder(EMBED_DIM, IMG_SIZE, PATCH_SIZE, NUM_CLASSES).to(DEVICE)
    vit.load_state_dict(ckpt["vit"], strict=False)
    dec.load_state_dict(ckpt["dec"], strict=False)
    vit.eval()
    dec.eval()
    print("Model loaded successfully.")

    dice_scores = []

    with torch.no_grad():
        for x, y in test_dl:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            # logits = predict_tta(vit, dec, x)
            logits = dec(vit(x))  # without TTA
            dice = dice_coef(logits, y).item()
            dice_scores.append(dice)

    mean_dice = float(np.mean(dice_scores)) if dice_scores else 0.0
    print(f"\n✅ Mean Dice: {mean_dice:.4f}")


# ============================================================
#                     MAIN
# ============================================================




Device: cuda


In [ ]:
if __name__ == "__main__":
    # 1) MIM pretraining
    # run_mim_pretraining()

    # 2) Segmentation training
    run_segmentation_training()

    # 3) Test with TTA
    run_test()


========== MIM PRETRAINING ==========
[MIM] epoch 1/50 | train_loss=0.408370
[MIM] epoch 2/50 | train_loss=0.059204
[MIM] epoch 3/50 | train_loss=0.033247
[MIM] epoch 4/50 | train_loss=0.025772
[MIM] epoch 5/50 | train_loss=0.019529
[MIM] epoch 6/50 | train_loss=0.018353
[MIM] epoch 7/50 | train_loss=0.015090
[MIM] epoch 8/50 | train_loss=0.014307
[MIM] epoch 9/50 | train_loss=0.011300
[MIM] epoch 10/50 | train_loss=0.010670
[MIM] epoch 11/50 | train_loss=0.009480
[MIM] epoch 12/50 | train_loss=0.008822
[MIM] epoch 13/50 | train_loss=0.008217
[MIM] epoch 14/50 | train_loss=0.007391
[MIM] epoch 15/50 | train_loss=0.007101
[MIM] epoch 16/50 | train_loss=0.005946
[MIM] epoch 17/50 | train_loss=0.005505
[MIM] epoch 18/50 | train_loss=0.004892
[MIM] epoch 19/50 | train_loss=0.004987
[MIM] epoch 20/50 | train_loss=0.004131
[MIM] epoch 21/50 | train_loss=0.004034
[MIM] epoch 22/50 | train_loss=0.003878
[MIM] epoch 23/50 | train_loss=0.003466
[MIM] epoch 24/50 | train_loss=0.003548
[MIM] epoc